In [ ]:
# Kitsune - ANN-based Anomaly Detection (Codespace)
# xNIDS: Explaining Deep Learning-based Network Intrusion Detection Systems
#
# Dataset: OS-Scan traffic (Kitsune / KitNET reimplementation)
# Model: Autoencoder (256->128->64->128->256), 115 network-stat features
# Mode: Inference with pretrained weights - ../Models/kitsune.h5
#
# This notebook demonstrates anomaly detection on pre-selected OS-Scan network
# traffic windows (true positives, false negatives, false positives) using the
# pretrained Kitsune autoencoder model.
#
# Option 2 (Research Paper): This reproduces the xNIDS baseline (USENIX Security
# 2023) on GitHub Codespaces and serves as the foundation for improvement and evaluation.

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from tensorflow import keras

print("TensorFlow/Keras loaded.")


In [ ]:
# 1. Load Pretrained Kitsune Model
# Load the autoencoder trained on 1M OS-Scan traffic samples.
# Anomaly score = per-sample MSE reconstruction error.

In [ ]:
model_path = "../Models/kitsune.h5"
autoencoder = keras.models.load_model(model_path)
autoencoder.summary()


In [ ]:
# 2. Model Architecture and Feature Groups
# The Kitsune autoencoder uses 115 features derived from 5 groups:
# - MAC-IP (15): packet sizes from a MAC/IP pair
# - IP (15): packet sizes from an IP address
# - Jitter (15): jitter from an IP address
# - IPtoIP (35): packet sizes between two IPs
# - Socket (35): packet sizes between two sockets
#
# Each group captures 23 statistics over 5 time windows.
# Anomaly threshold (from training): 0.20776055640832747

In [ ]:
# Kitsune feature groups (115 features total across 5 network-stat groups)
kitsune_group_sizes = [15, 15, 15, 35, 35]
kitsune_group_names = ["MAC-IP", "IP", "Jitter", "IPtoIP", "Socket"]
kitsune_feature_names = [f'f{i}' for i in range(1, 116)]

# Anomaly threshold derived from training on 1M OS-Scan samples
# threshold = mean(reconstruction_error) + 2 * std(reconstruction_error)
THRESHOLD = 0.20776055640832747

print(f"Feature groups: {list(zip(kitsune_group_names, kitsune_group_sizes))}")
print(f"Total features : {sum(kitsune_group_sizes)}")
print(f"Anomaly threshold: {THRESHOLD:.6f}")


In [ ]:
# 3. Load Pre-selected OS-Scan Sample Windows
# These sliding windows (10 consecutive normalized packets each) were extracted
# from the original test set and committed to Data/.
# - TP: correctly flagged attack traffic (index 373907, error ~= 0.267)
# - FN: attack traffic missed by the model (index 456016, error ~= 0.142)
# - FP: benign traffic incorrectly flagged (index 373081, error ~= 0.303)

In [ ]:
tp_data = pd.read_csv("../Data/kitsune_selected_tp_rows.csv")
fn_data = pd.read_csv("../Data/kitsune_selected_fn_rows.csv")
fp_data = pd.read_csv("../Data/kitsune_selected_fp_rows.csv")

print(f"TP window shape : {tp_data.shape}")
print(f"FN window shape : {fn_data.shape}")
print(f"FP window shape : {fp_data.shape}")
tp_data.head(3)


In [ ]:
# 4. Compute Reconstruction Error on Selected Windows
# Run each window through the autoencoder and compute per-sample MSE.
# Compare against the anomaly threshold.

In [ ]:
def reconstruction_error(model, data):
    """Compute per-sample MSE reconstruction error."""
    X = data.to_numpy() if hasattr(data, 'to_numpy') else data
    X_reconstructed = model.predict(X, verbose=0)
    return np.mean(np.square(X - X_reconstructed), axis=1)

tp_errors = reconstruction_error(autoencoder, tp_data)
fn_errors = reconstruction_error(autoencoder, fn_data)
fp_errors = reconstruction_error(autoencoder, fp_data)

print(f"TP reconstruction errors: {np.round(tp_errors, 5)}")
print(f"FN reconstruction errors: {np.round(fn_errors, 5)}")
print(f"FP reconstruction errors: {np.round(fp_errors, 5)}")


In [ ]:
# 5. Anomaly Detection on Selected Windows
# Classify each sample as anomaly (1) or benign (0) by comparing
# reconstruction error against THRESHOLD.

In [ ]:
def classify(errors, threshold=THRESHOLD):
    return (errors > threshold).astype(int)

tp_preds = classify(tp_errors)
fn_preds = classify(fn_errors)
fp_preds = classify(fp_errors)

summary = pd.DataFrame({
    "Window": ["TP (attack, should=1)", "FN (attack, should=1)", "FP (benign, should=0)"],
    "Last-sample error": [tp_errors[-1], fn_errors[-1], fp_errors[-1]],
    "Last-sample predicted": [tp_preds[-1], fn_preds[-1], fp_preds[-1]],
    "Expected label": [1, 1, 0],
    "Correct?": [tp_preds[-1] == 1, fn_preds[-1] == 1, fp_preds[-1] == 0],
})
print(f"Anomaly threshold: {THRESHOLD:.6f}\n")
print(summary.to_string(index=False))


In [ ]:
# 6. Visualize Reconstruction Errors
# Plot reconstruction error across all three windows. The red dashed line marks
# the anomaly threshold. Samples above it are classified as attacks.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4), sharey=True)

cases = [
    ("True Positive\n(attack correctly detected)", tp_errors, "tab:green"),
    ("False Negative\n(attack missed)", fn_errors, "tab:orange"),
    ("False Positive\n(benign wrongly flagged)", fp_errors, "tab:red"),
]

for ax, (title, errors, color) in zip(axes, cases):
    ax.plot(errors, marker='o', color=color, linewidth=1.5, markersize=4)
    ax.axhline(y=THRESHOLD, color='red', linestyle='--', linewidth=1.2, label=f'Threshold ({THRESHOLD:.3f})')
    ax.set_title(title, fontsize=10)
    ax.set_xlabel("Sample index in window")
    ax.set_ylabel("Reconstruction Error (MSE)")
    ax.legend(fontsize=8)

fig.suptitle("Kitsune Reconstruction Error — OS-Scan Selected Windows", fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()


In [ ]:
# 7. Feature Group Contribution (Reconstruction Error per Group)
# Decompose reconstruction error of the final sample in each window by feature
# group. This highlights which network-stat group drives the anomaly score.

In [ ]:
def group_reconstruction_error(model, sample_row, group_sizes):
    """Per-group MSE: reconstruct full sample then slice error by group."""
    x = sample_row.to_numpy().reshape(1, -1)
    x_hat = model.predict(x, verbose=0)
    sq_err = np.square(x - x_hat)[0]
    group_errors = []
    idx = 0
    for size in group_sizes:
        group_errors.append(np.mean(sq_err[idx:idx+size]))
        idx += size
    return group_errors

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
cases = [
    ("True Positive (last sample)", tp_data.iloc[-1], "tab:green"),
    ("False Negative (last sample)", fn_data.iloc[-1], "tab:orange"),
    ("False Positive (last sample)", fp_data.iloc[-1], "tab:red"),
]

for ax, (title, row, color) in zip(axes, cases):
    g_errors = group_reconstruction_error(autoencoder, row, kitsune_group_sizes)
    bars = ax.bar(kitsune_group_names, g_errors, color=color, alpha=0.75)
    ax.axhline(y=THRESHOLD, color='red', linestyle='--', linewidth=1.0, label='Threshold')
    ax.set_title(title, fontsize=9)
    ax.set_xlabel("Feature Group")
    ax.set_ylabel("Mean Reconstruction Error")
    ax.legend(fontsize=8)

fig.suptitle("Per-Group Reconstruction Error for Each Window Type", fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()


In [ ]:
# 8. Summary and Handoff to explanation.ipynb
# The selected windows (kitsune_selected_tp/fn/fp_rows.csv) are consumed by
# explanation.ipynb to generate per-feature attribution scores via sparse-group LASSO.
# Known scores from original run:
# - TP: 0.26677650458466096
# - FN: 0.14151900999030606
# - FP: 0.30298337954205980

In [ ]:
# Prediction scores recorded during original training run (used by explanation.ipynb)
KITSUNE_SCORES = {
    "true_positive":  {"data_path": "../Data/kitsune_selected_tp_rows.csv", "score": 0.26677650458466096},
    "false_negative": {"data_path": "../Data/kitsune_selected_fn_rows.csv", "score": 0.14151900999030606},
    "false_positive": {"data_path": "../Data/kitsune_selected_fp_rows.csv", "score": 0.3029833795420598},
}

print("Kitsune inference config ready for explanation.ipynb:\n")
for k, v in KITSUNE_SCORES.items():
    actual_error = reconstruction_error(autoencoder,
                       pd.read_csv(v["data_path"]).iloc[[-1]])
    print(f"  {k:15s} | stored score={v['score']:.5f} | live error={actual_error[0]:.5f} | "
          f"anomaly={'YES' if actual_error[0] > THRESHOLD else 'NO'}")
